# Part 4e — Policy instruments

### Government constrains, firms respond

This closes the loop opened in Part 4a: **government sets non-market constraints; firms optimise
within them.** Every lever here is exogenous and swept — never chosen by the model. Endogenising the
policy-setter would make this a trilevel program, which is a research project rather than a teaching
notebook.

Three instruments, all live in critical-minerals policy:

| Lever | Mechanism | Implementation |
|---|---|---|
| **Tariff** | per-unit duty on imports | adder on the seller's transport cost |
| **Quota** | cap on imports into a market | upper bound on cross-region sales |
| **Local content** | floor on domestic supply | lower bound on the home firm's own-market sales |

The behavioural model is the Cournot game of Part 4c, with a Stackelberg section at the end using the
MPEC of Part 4d.

### Welfare accounting

A policy question needs a welfare metric, not just profit. We report

$$W = \underbrace{\text{CS}}_{\text{consumer surplus}} + \underbrace{\sum_r \Pi_r}_{\text{producer profit}} + \underbrace{T}_{\text{tariff revenue}}$$

with consumer surplus the area under inverse demand above price, $\tfrac{1}{2}B Q^2$. Every term is
discounted with $\omega_p$.

This is deliberately a **narrow** welfare measure. It ignores everything a real minerals policy is
usually *for*: supply security, employment, strategic autonomy, emissions. A tariff that reduces $W$
here may still be justified on grounds this model cannot see. Read the numbers as "what it costs in
market efficiency," not "whether it is a good idea."

## 1. Setup

In [ ]:
!pip install gurobipy --quiet
import os, math
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})
ENV = None          # set to gp.Env(params=...) for a full WLS licence
print("gurobipy", gp.gurobi.version())

In [ ]:
REGIONS = ['R1', 'R2']
STAGES  = ['MINE', 'PROC', 'MFG']

# ---------------- TIME ----------------
import os
# P4_SMALL shrinks the horizon so the EXACT MIQP fits the size-limited licence,
# letting us validate the piecewise-linear revenue against the true quadratic.
BLOCKS = ([(2, 1), (1, 3)] if os.environ.get('P4_SMALL') else [(6, 1), (4, 3), (2, 5), (1, 9)])
LEN, START = [], []
_y = 1
for _c, _L in BLOCKS:
    for _ in range(_c):
        LEN.append(_L); START.append(_y); _y += _L
P = list(range(len(LEN)))
HORIZON = _y - 1
YEARS = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
DR = 0.05
OMEGA = {p: sum(1/(1+DR)**t for t in YEARS[p]) for p in P}
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}
REPORT_UNTIL = 28

# ---------------- TECH ----------------
LIFE = 25
LEAD = {'MINE': 1, 'PROC': 2, 'MFG': 2}
CAP_MIN, CAP_MAX = 60.0, 260.0
CRF = DR*(1+DR)**LIFE/((1+DR)**LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}
MU = {(s, v): CRF*sum(1/(1+DR)**t for t in range(ONLINE[s, v], ONLINE[s, v]+LIFE)
                      if t <= HORIZON) for s in STAGES for v in P}

In [ ]:
# ---------------- ASYMMETRIC INSTANCE ----------------
# R1 = incumbent upstream processor with accumulated experience.
# R2 = entrant, cheaper to build, trying to move downstream.
FIXED = {('MINE','R1'):900.,('PROC','R1'):1500.,('MFG','R1'):1300.,
         ('MINE','R2'):820.,('PROC','R2'):1350.,('MFG','R2'):1180.}
UNIT  = {('MINE','R1'):7.0,('PROC','R1'):11.0,('MFG','R1'):9.5,
         ('MINE','R2'):6.4,('PROC','R2'):10.0,('MFG','R2'):8.7}
OPEX  = {('MINE','R1'):1.2,('PROC','R1'):2.0,('MFG','R1'):2.4,
         ('MINE','R2'):1.35,('PROC','R2'):2.2,('MFG','R2'):2.6}

LEGACY_CAP = {('MINE','R1'):230,('PROC','R1'):205,('MFG','R1'):155,
              ('MINE','R2'):165,('PROC','R2'):125,('MFG','R2'):100}
LEGACY_RET = {('MINE','R1'):11,('PROC','R1'):14,('MFG','R1'):18,
              ('MINE','R2'):9, ('PROC','R2'):16,('MFG','R2'):22}
LEGACY_BYR = -8
# incumbent starts with accumulated production experience
EXPERIENCE0 = {'R1': 2600.0, 'R2': 500.0}

In [ ]:
# ---------------- EFFICIENCY (yield) ----------------
ETA_CEIL = {'MINE':0.92,'PROC':0.95,'MFG':0.93}
ETA_BASE = {'MINE':0.86,'PROC':0.80,'MFG':0.78}
ALPHA    = {'MINE':0.0,'PROC':0.030,'MFG':0.025}
BETA     = {'MINE':0.0,'PROC':0.010,'MFG':0.008}
DELTA_BAR= {'MINE':0.02,'PROC':0.05,'MFG':0.05}
ETA_FLOOR= 0.60
VINTAGES = [-1] + P
BYEAR = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}
ETA = {}
for s in STAGES:
    for v in VINTAGES:
        fr = ETA_CEIL[s] - (ETA_CEIL[s]-ETA_BASE[s])*(1-ALPHA[s])**(BYEAR[v]-1)
        fr = max(ETA_FLOOR, min(fr, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p]-BYEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s]-fr)*(1-BETA[s])**age
            ETA[s, v, p] = max(ETA_FLOOR, min(fr+DELTA_BAR[s], aged))

In [ ]:
# ---------------- DEMAND & MARKET ----------------
DEMAND = {}
for r, base, g in [('R1', 100.0, 0.008), ('R2', 75.0, 0.026)]:
    for p in P:
        DEMAND[r, p] = sum(base*(1+g)**(t-1) for t in YEARS[p])/LEN[p]
TRANSPORT = {(rf, rt): (0.5 if rf == rt else 2.4) for rf in REGIONS for rt in REGIONS}
PRICE_FIXED = 12.0

# ---------------- GOVERNMENT LEVERS (exogenous, swept) ----------------
# TARIFF[(seller_region, market)] : per-unit duty on imports into `market`
# QUOTA [(seller_region, market)] : per-period cap on imports into `market`
# LOCAL_MIN[market]               : per-period floor on the DOMESTIC firm's own-market sales
TARIFF, QUOTA, LOCAL_MIN = {}, {}, {}

def clear_policy():
    TARIFF.clear(); QUOTA.clear(); LOCAL_MIN.clear()

def set_tariff(rate, on_imports_to=None):
    TARIFF.clear()
    for rf in REGIONS:
        for rt in REGIONS:
            if rf != rt and (on_imports_to is None or rt == on_imports_to):
                TARIFF[rf, rt] = rate

def set_quota(cap, on_imports_to=None):
    QUOTA.clear()
    for rf in REGIONS:
        for rt in REGIONS:
            if rf != rt and (on_imports_to is None or rt == on_imports_to):
                QUOTA[rf, rt] = cap

def set_local_min(level, market=None):
    LOCAL_MIN.clear()
    for rt in REGIONS:
        if market is None or rt == market:
            LOCAL_MIN[rt] = level
PEN_SHORT, PEN_DISPOSE = 90.0, 12.0

In [ ]:
# ---------------- LEARNING ----------------
LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX, Q_START, Q_ADD, CAPEX_FLOOR, NBP = 0.15, 300.0, 700.0, 0.60, 9
_bc = -math.log2(1-LR_CAPEX)
K = list(range(NBP))
QBP = [Q_START + Q_ADD*k/(NBP-1) for k in K]

def _cap_unit_mult(q):
    return max(CAPEX_FLOOR, (q/Q_START)**(-_bc))

def _cap_cum_mult(q, n=400):
    if q <= Q_START:
        return 0.0
    h = (q-Q_START)/n
    return sum(0.5*(_cap_unit_mult(Q_START+i*h)+_cap_unit_mult(Q_START+(i+1)*h))*h
               for i in range(n))
CBP = [_cap_cum_mult(q) for q in QBP]

LR_OPEX, OPEX_FLOOR, LAG_YEARS, N_TIERS = 0.18, 0.65, 3, 3
TIER_Q, TIER_M = {}, {}

def set_tiers(top_by_region):
    for r in REGIONS:
        top = max(top_by_region[r], 1.0)
        q1 = top/8.0
        TIER_Q[r] = [q1*2**j for j in range(N_TIERS-1)]
        TIER_M[r] = [max(OPEX_FLOOR, (1-LR_OPEX)**j) for j in range(N_TIERS)]

ACTIVE = {r: [(s, v, p) for s in STAGES for v in VINTAGES for p in P
              if (v == -1 and START[p] <= LEGACY_RET[s, r])
              or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v]+LIFE-1)]
          for r in REGIONS}
VIN = {(r, s, p): [v for (ss, v, pp) in ACTIVE[r] if (ss, pp) == (s, p)]
       for r in REGIONS for s in STAGES for p in P}
BUILD = {r: [(s, v) for s in STAGES for v in P if ONLINE[s, v] <= HORIZON]
         for r in REGIONS}

## 2. The levers

All three default to **off**, so Parts 4a–4d are unaffected by their presence. A tariff enters the
seller's cost expression; a quota and a local content floor enter as bounds on its sales.

Note who pays what. The tariff is a cost to the **exporting firm** and revenue to the **importing
government** — so it appears twice in the welfare sum with opposite signs, and nets out except for
the behavioural change it induces. That is the whole reason a tariff and a quota differ.

In [ ]:
# ---------------- GOVERNMENT LEVERS (exogenous, swept) ----------------
# TARIFF[(seller_region, market)] : per-unit duty on imports into `market`
# QUOTA [(seller_region, market)] : per-period cap on imports into `market`
# LOCAL_MIN[market]               : per-period floor on the DOMESTIC firm's own-market sales
TARIFF, QUOTA, LOCAL_MIN = {}, {}, {}

def clear_policy():
    TARIFF.clear(); QUOTA.clear(); LOCAL_MIN.clear()

def set_tariff(rate, on_imports_to=None):
    TARIFF.clear()
    for rf in REGIONS:
        for rt in REGIONS:
            if rf != rt and (on_imports_to is None or rt == on_imports_to):
                TARIFF[rf, rt] = rate

def set_quota(cap, on_imports_to=None):
    QUOTA.clear()
    for rf in REGIONS:
        for rt in REGIONS:
            if rf != rt and (on_imports_to is None or rt == on_imports_to):
                QUOTA[rf, rt] = cap

def set_local_min(level, market=None):
    LOCAL_MIN.clear()
    for rt in REGIONS:
        if market is None or rt == market:
            LOCAL_MIN[rt] = level
PEN_SHORT, PEN_DISPOSE = 90.0, 12.0

# ---------------- LEARNING ----------------
LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX, Q_START, Q_ADD, CAPEX_FLOOR, NBP = 0.15, 300.0, 700.0, 0.60, 9
_bc = -math.log2(1-LR_CAPEX)
K = list(range(NBP))
QBP = [Q_START + Q_ADD*k/(NBP-1) for k in K]

def _cap_unit_mult(q):
    return max(CAPEX_FLOOR, (q/Q_START)**(-_bc))

def _cap_cum_mult(q, n=400):
    if q <= Q_START:
        return 0.0
    h = (q-Q_START)/n
    return sum(0.5*(_cap_unit_mult(Q_START+i*h)+_cap_unit_mult(Q_START+(i+1)*h))*h
               for i in range(n))
CBP = [_cap_cum_mult(q) for q in QBP]

LR_OPEX, OPEX_FLOOR, LAG_YEARS, N_TIERS = 0.18, 0.65, 3, 3
TIER_Q, TIER_M = {}, {}

def set_tiers(top_by_region):
    for r in REGIONS:
        top = max(top_by_region[r], 1.0)
        q1 = top/8.0
        TIER_Q[r] = [q1*2**j for j in range(N_TIERS-1)]
        TIER_M[r] = [max(OPEX_FLOOR, (1-LR_OPEX)**j) for j in range(N_TIERS)]

In [ ]:
def add_region(m, r, learning='both'):
    """Attach one region's vertically-integrated chain to model m. Returns handles."""
    b = m.addVars(BUILD[r], vtype=GRB.BINARY, name=f'b_{r}')
    c = m.addVars(BUILD[r], lb=0.0, ub=CAP_MAX, name=f'c_{r}')
    x = m.addVars(ACTIVE[r], lb=0.0, name=f'x_{r}')
    f_mp = m.addVars(P, lb=0.0, name=f'fmp_{r}')
    f_pf = m.addVars(P, lb=0.0, name=f'fpf_{r}')
    sale = m.addVars(REGIONS, P, lb=0.0, name=f'sale_{r}')
    disp = m.addVars(P, lb=0.0, name=f'disp_{r}')

    m.addConstrs((c[s, v] <= CAP_MAX*b[s, v] for (s, v) in BUILD[r]), name=f'su_{r}')
    m.addConstrs((c[s, v] >= CAP_MIN*b[s, v] for (s, v) in BUILD[r]), name=f'sl_{r}')
    m.addConstrs((x[s, v, p] <= (LEGACY_CAP[s, r] if v == -1 else c[s, v])
                  for (s, v, p) in ACTIVE[r]), name=f'cap_{r}')
    m.addConstrs((gp.quicksum(ETA['MINE', v, p]*x['MINE', v, p]
                              for v in VIN[r, 'MINE', p]) == f_mp[p] for p in P),
                 name=f'mine_{r}')
    m.addConstrs((f_mp[p] == gp.quicksum(x['PROC', v, p] for v in VIN[r, 'PROC', p])
                  for p in P), name=f'pin_{r}')
    m.addConstrs((gp.quicksum(ETA['PROC', v, p]*x['PROC', v, p]
                              for v in VIN[r, 'PROC', p]) == f_pf[p] for p in P),
                 name=f'pout_{r}')
    m.addConstrs((f_pf[p] == gp.quicksum(x['MFG', v, p] for v in VIN[r, 'MFG', p])
                  for p in P), name=f'min_{r}')
    m.addConstrs((gp.quicksum(ETA['MFG', v, p]*x['MFG', v, p]
                              for v in VIN[r, 'MFG', p])
                  == sale.sum('*', p) + disp[p] for p in P), name=f'mout_{r}')

    # cumulative production (undiscounted), regional scope, with initial experience
    cum = m.addVars(P, lb=0.0, ub=3*CAP_MAX*HORIZON + EXPERIENCE0[r], name=f'cum_{r}')
    m.addConstrs((cum[p] == EXPERIENCE0[r] +
                  gp.quicksum(LEN[q]*x['MFG', v, q] for q in P if q <= p
                              for v in VIN[r, 'MFG', q]) for p in P), name=f'cp_{r}')

    capex = gp.quicksum(MU[s, v]*FIXED[s, r]*b[s, v] for (s, v) in BUILD[r]) \
          + gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                        for (s, v) in BUILD[r] if s not in LEARN_STAGES)
    if learning in ('capacity', 'both'):
        Q = m.addVars(P, lb=Q_START, ub=Q_START+Q_ADD, name=f'Q_{r}')
        Cc = m.addVars(P, lb=0.0, name=f'C_{r}')
        lam = m.addVars(P, K, lb=0.0, ub=1.0, name=f'lam_{r}')
        m.addConstrs((lam.sum(p, '*') == 1 for p in P), name=f'sc_{r}')
        m.addConstrs((Q[p] == gp.quicksum(QBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sQ_{r}')
        m.addConstrs((Cc[p] == gp.quicksum(CBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sC_{r}')
        m.addConstrs((Q[p] == Q_START + gp.quicksum(c[s, v] for (s, v) in BUILD[r]
                                                    if s in LEARN_STAGES and v <= p)
                      for p in P), name=f'cc_{r}')
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])
        rate = sum(UNIT[s, r] for s in LEARN_STAGES)/len(LEARN_STAGES)
        capex += gp.quicksum(MU['PROC', p]*rate*(Cc[p]-(Cc[p-1] if p > 0 else 0.0))
                             for p in P)
    else:
        capex += gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                             for (s, v) in BUILD[r] if s in LEARN_STAGES)

    if learning in ('production', 'both') and TIER_Q:
        J = list(range(N_TIERS))
        z = m.addVars(P, J, vtype=GRB.BINARY, name=f'z_{r}')
        m.addConstrs((z.sum(p, '*') == 1 for p in P), name=f'ot_{r}')
        LAGP = {p: YEAR_TO_P[max(1, START[p]-LAG_YEARS)] for p in P}
        BIGQ = 3*CAP_MAX*HORIZON + EXPERIENCE0[r]
        m.addConstrs((cum[LAGP[p]] >= TIER_Q[r][j-1] - BIGQ*(1-z[p, j])
                      for p in P for j in J if j > 0), name=f'tf_{r}')
        m.addConstrs((cum[LAGP[p]] <= TIER_Q[r][j] + BIGQ*(1-z[p, j])
                      for p in P for j in J if j < N_TIERS-1), name=f'tc_{r}')
        ts = m.addVars(STAGES, P, J, lb=0.0, name=f'ts_{r}')
        m.addConstrs((ts.sum(s, p, '*') == gp.quicksum(x[s, v, p] for v in VIN[r, s, p])
                      for s in STAGES for p in P), name=f'tss_{r}')
        m.addConstrs((ts[s, p, j] <= 3*CAP_MAX*z[p, j]
                      for s in STAGES for p in P for j in J), name=f'tl_{r}')
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*TIER_M[r][j]*ts[s, p, j]
                           for s in STAGES for p in P for j in J)
    else:
        z = None
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*x[s, v, p] for (s, v, p) in ACTIVE[r])

    m.addConstrs((sale[rt, p] <= QUOTA[r, rt] for rt in REGIONS for p in P
                  if (r, rt) in QUOTA), name=f'quota_{r}')
    m.addConstrs((sale[r, p] >= LOCAL_MIN[r] for p in P
                  if r in LOCAL_MIN), name=f'lcr_{r}')
    trans = gp.quicksum(OMEGA[p]*(TRANSPORT[r, rt] + TARIFF.get((r, rt), 0.0))*sale[rt, p]
                        for rt in REGIONS for p in P)
    tariff_paid = gp.quicksum(OMEGA[p]*TARIFF.get((r, rt), 0.0)*sale[rt, p]
                              for rt in REGIONS for p in P)
    dcost = gp.quicksum(OMEGA[p]*PEN_DISPOSE*disp[p] for p in P)
    revenue = gp.quicksum(OMEGA[p]*PRICE_FIXED*sale[rt, p] for rt in REGIONS for p in P)
    return dict(b=b, c=c, x=x, sale=sale, disp=disp, cum=cum, z=z,
                capex=capex, opex=opex, trans=trans, dcost=dcost, revenue=revenue,
                tariff_paid=tariff_paid, cost=capex+opex+trans+dcost)

In [ ]:
def solve_planner(w1=0.5, learning='both', mipgap=0.005, quiet=True):
    m = gp.Model(); m.Params.OutputFlag = 0 if quiet else 1; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    short = m.addVars(REGIONS, P, lb=0.0, name='short')
    m.addConstrs((gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS) + short[rt, p]
                  >= DEMAND[rt, p] for rt in REGIONS for p in P), name='demand')
    pen = gp.quicksum(OMEGA[p]*PEN_SHORT*short[rt, p] for rt in REGIONS for p in P)
    m.setObjective(w1*H['R1']['cost'] + (1-w1)*H['R2']['cost'] + pen, GRB.MINIMIZE)
    m.optimize()
    m._H, m._short, m._pen = H, short, pen
    return m

In [ ]:
# ================= 4c: Cournot with endogenous price =================
CHOKE    = 30.0     # price at zero quantity
P_ANCHOR = 13.0     # price when quantity equals the Part 4b demand reference
A_INT = {(rt, p): CHOKE for rt in REGIONS for p in P}
B_SLP = {(rt, p): (CHOKE - P_ANCHOR) / DEMAND[rt, p] for rt in REGIONS for p in P}


NBP_REV = 7          # breakpoints for the piecewise-linear revenue curve


def _rev_breakpoints(a_eff, b, smax, n=NBP_REV):
    """Breakpoints for revenue(s) = a_eff*s - b*s^2 on [0, smax].
    The function is CONCAVE and we MAXIMISE, so the chord between any two
    breakpoints lies BELOW the curve. A free convex combination therefore has no
    incentive to mix non-adjacent points -- unlike the concave-MINIMISE case in
    Part 3, this needs no SOS2 and adds no binaries."""
    S = [smax * k / (n - 1) for k in range(n)]
    R = [a_eff * x - b * x * x for x in S]
    return S, R

In [ ]:
NBP_REV = 7          # breakpoints for the piecewise-linear revenue curve


def _rev_breakpoints(a_eff, b, smax, n=NBP_REV):
    """Breakpoints for revenue(s) = a_eff*s - b*s^2 on [0, smax].
    The function is CONCAVE and we MAXIMISE, so the chord between any two
    breakpoints lies BELOW the curve. A free convex combination therefore has no
    incentive to mix non-adjacent points -- unlike the concave-MINIMISE case in
    Part 3, this needs no SOS2 and adds no binaries."""
    S = [smax * k / (n - 1) for k in range(n)]
    R = [a_eff * x - b * x * x for x in S]
    return S, R

In [ ]:
def best_response_cournot(r, rival_sales, learning='both', mipgap=0.005):
    """Firm r maximises profit facing linear inverse demand
       p[rt,p] = A - B*(own + rival).
    Revenue is piecewise-linearised in own quantity, keeping the model a MILP."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    h = add_region(m, r, learning)
    s = h['sale']
    KR = list(range(NBP_REV))
    mu = m.addVars(REGIONS, P, KR, lb=0.0, ub=1.0, name='mu')
    rev_t = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            q_bar = rival_sales.get((rt, p), 0.0)
            a_eff = A_INT[rt, p] - B_SLP[rt, p] * q_bar
            smax = max(1e-6, A_INT[rt, p] / B_SLP[rt, p] - q_bar)
            S, R = _rev_breakpoints(a_eff, B_SLP[rt, p], smax, len(KR))
            m.addConstr(mu.sum(rt, p, '*') == 1, name=f'rcvx_{rt}_{p}')
            m.addConstr(s[rt, p] == gp.quicksum(S[k] * mu[rt, p, k] for k in KR),
                        name=f'rS_{rt}_{p}')
            m.addConstr(rev_t[rt, p] == gp.quicksum(R[k] * mu[rt, p, k] for k in KR),
                        name=f'rR_{rt}_{p}')
    revenue = gp.quicksum(OMEGA[p] * rev_t[rt, p] for rt in REGIONS for p in P)
    m.setObjective(revenue - h['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h, m._rev = h, revenue
    return m

In [ ]:
def cournot_iterate(learning='both', first='R1', max_iter=16, tol=0.5, mipgap=1e-3):
    """Iterated best response under Cournot competition.

    Convergence for a game with CONTINUOUS strategies must be tested with a
    TOLERANCE, not by exact state matching: each best response is a MILP solved to
    a finite gap, so the returned quantities wobble slightly between iterations.
    Exact hashing reads that wobble as a cycle."""
    def dist(a, b):
        return max(abs(a[r][k] - b[r][k]) for r in REGIONS for k in a[r])

    sales = {r: {(rt, p): 0.0 for rt in REGIONS for p in P} for r in REGIONS}
    plans, hist, log = {}, [], []
    order = [first, 'R2' if first == 'R1' else 'R1']
    for it in range(max_iter):
        prev = {r: dict(sales[r]) for r in REGIONS}
        for r in order:
            other = 'R2' if r == 'R1' else 'R1'
            m = best_response_cournot(r, sales[other], learning=learning, mipgap=mipgap)
            if m.SolCount == 0:
                return dict(status='INFEASIBLE', iters=it, log=log)
            sales[r] = {(rt, p): m._h['sale'][rt, p].X for rt in REGIONS for p in P}
            plans[r] = tuple(sorted((s_, v) for (s_, v) in m._h['b']
                                    if m._h['b'][s_, v].X > 0.5))
            log.append(dict(iter=it, firm=r, profit=m.ObjVal,
                            revenue=m._rev.getValue(), cost=m._h['cost'].getValue(),
                            builds=len(plans[r]), sales=sum(sales[r].values()),
                            disposal=sum(m._h['disp'][p].X for p in P)))
        cur = {r: dict(sales[r]) for r in REGIONS}
        if it > 0 and dist(cur, prev) < tol:
            return dict(status='CONVERGED', cycle_len=1, iters=it + 1, log=log,
                        plans=plans, sales=sales, drift=dist(cur, prev))
        for k, past in enumerate(hist):                      # genuine k-cycle, k >= 2
            if dist(cur, past) < tol:
                return dict(status='CYCLE', cycle_len=len(hist) - k, iters=it + 1,
                            log=log, plans=plans, sales=sales)
        hist.append(cur)
    return dict(status='MAX_ITER', iters=max_iter, log=log, plans=plans, sales=sales)

In [ ]:
def market_outcome(sales):
    rows = []
    for rt in REGIONS:
        for p in P:
            q = sum(sales[r][rt, p] for r in REGIONS)
            price = A_INT[rt, p] - B_SLP[rt, p] * q
            rows.append(dict(market=rt, period=p, year=START[p], quantity=q, price=price,
                             consumer_surplus=0.5 * B_SLP[rt, p] * q * q,
                             share_R1=(sales['R1'][rt, p] / q if q > 1e-6 else None)))
    return rows

In [ ]:
def joint_profit_max(learning='both', mipgap=0.005):
    """Collusive benchmark: one decision maker maximising the SUM of both profits."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    KR = list(range(NBP_REV))
    mu = m.addVars(REGIONS, P, KR, lb=0.0, ub=1.0, name='mu')
    rev_t = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            smax = A_INT[rt, p] / B_SLP[rt, p]
            S, R = _rev_breakpoints(A_INT[rt, p], B_SLP[rt, p], smax, len(KR))
            m.addConstr(mu.sum(rt, p, '*') == 1)
            m.addConstr(gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS)
                        == gp.quicksum(S[k] * mu[rt, p, k] for k in KR))
            m.addConstr(rev_t[rt, p] == gp.quicksum(R[k] * mu[rt, p, k] for k in KR))
    revenue = gp.quicksum(OMEGA[p] * rev_t[rt, p] for rt in REGIONS for p in P)
    m.setObjective(revenue - gp.quicksum(H[r]['cost'] for r in REGIONS), GRB.MAXIMIZE)
    m.optimize()
    m._H, m._rev = H, revenue
    return m

In [ ]:
def welfare(sales, profits):
    """Consumer surplus + firm profits + tariff revenue, all discounted."""
    cs = sum(OMEGA[p]*0.5*B_SLP[rt, p]*(sum(sales[r][rt, p] for r in REGIONS))**2
             for rt in REGIONS for p in P)
    tr = sum(OMEGA[p]*TARIFF.get((r, rt), 0.0)*sales[r][rt, p]
             for r in REGIONS for rt in REGIONS for p in P)
    return dict(consumer_surplus=cs, tariff_revenue=tr,
                producer_profit=sum(profits.values()),
                total=cs + tr + sum(profits.values()))

In [ ]:
m0 = solve_planner(0.5, learning='capacity')
top = {r: m0._H[r]['cum'][P[-1]].X for r in REGIONS}
set_tiers(top)

def run_policy(tag):
    r = cournot_iterate(first='R1', max_iter=16)
    last = {L['firm']: L for L in r['log'][-2:]}
    pr = {f: last[f]['profit'] for f in REGIONS}
    W = welfare(r['sales'], pr)
    mo = pd.DataFrame(market_outcome(r['sales']))
    return dict(policy=tag, R1_profit=round(pr['R1'], 1), R2_profit=round(pr['R2'], 1),
                R1_sales=round(last['R1']['sales'], 1),
                R2_sales=round(last['R2']['sales'], 1),
                avg_price=round(mo.price.mean(), 2),
                consumer_surplus=round(W['consumer_surplus'], 1),
                gov_revenue=round(W['tariff_revenue'], 1),
                welfare=round(W['total'], 1))
print("baseline ready")

## 3. Tariffs

R2 is the entrant; suppose its government protects its home market against R1's imports.

In [ ]:
rows = []
for t in [0.0, 2.0, 5.0, 9.0]:
    clear_policy(); set_tariff(t, on_imports_to='R2')
    rows.append(dict(tariff=t, **run_policy(f'tariff {t:.0f}')))
clear_policy()
dfT = pd.DataFrame(rows).drop(columns='policy'); dfT

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(dfT.tariff, dfT.R1_profit, 'o-', lw=2.5, color='#2471a3', label='R1 (exporter)')
ax[0].plot(dfT.tariff, dfT.R2_profit, 's-', lw=2.5, color='#d68910', label='R2 (protected)')
ax[0].set_xlabel('tariff on imports into R2'); ax[0].set_ylabel('profit')
ax[0].legend(); ax[0].set_title('Tariffs redistribute between producers')
ax[1].plot(dfT.tariff, dfT.consumer_surplus, 'o-', lw=2.5, color='#196f3d', label='consumer surplus')
ax[1].plot(dfT.tariff, dfT.welfare, 'D-', lw=2.5, color='#c0392b', label='total welfare')
ax[1].plot(dfT.tariff, dfT.gov_revenue, '^-', lw=2.5, color='#8e44ad', label='tariff revenue')
ax[1].set_xlabel('tariff on imports into R2'); ax[1].set_ylabel('discounted value')
ax[1].legend(fontsize=10); ax[1].set_title('...and shrink the pie')
plt.tight_layout(); plt.show()

The classic result, reproduced from a supply chain model rather than assumed:

- **The protected firm gains a great deal.** R2's profit rises from 6,713 to 11,124 — it captures
  more of its home market as R1's landed cost rises.
- **The exporter loses more than that.** R1 falls from 11,527 to 7,175.
- **Consumers lose heavily.** Price rises from 16.05 to 17.86 and consumer surplus falls 20,756 →
  15,772, because the tariff works precisely *by* making supply more expensive.
- **Total welfare falls monotonically** — 38,995 → 35,859. The deadweight loss exceeds the tariff
  revenue collected.
- **Tariff revenue plateaus.** Rising rates eventually shrink the import volume they are levied on,
  the Laffer-style turning point visible between 5 and 9.

If the objective were narrow market efficiency, no tariff would be justified. The case for one has to
rest on something outside this model — which is exactly why the levers are exogenous here.

## 4. Quotas — same protection, worse arithmetic

A quota restricts imports directly rather than pricing them. The comparison is the standard one:
under a tariff the government collects the rents created by scarcity; under a quota those rents go
to whoever holds the import licence — here, implicitly, to the firms through a higher price.

In [ ]:
rows = []
clear_policy(); rows.append(run_policy('no policy'))
for q in [60.0, 30.0, 10.0]:
    clear_policy(); set_quota(q, on_imports_to='R2')
    rows.append(run_policy(f'quota {q:.0f}'))
clear_policy()
pd.DataFrame(rows)

A quota of 10 delivers protection comparable to a tariff of 9 — R2's profit reaches 11,416 against
11,124 — but note the `gov_revenue` column: **zero**. The scarcity rent that a tariff would have
collected is simply not collected.

Welfare under the quota (35,410) is accordingly worse than under the tariff (35,859) at similar
protection levels. **Same distortion, less revenue.** That is the textbook argument for preferring
tariffs to quotas, and here it falls out of the model rather than being asserted.

## 5. Local content — a lever that can backfire

A local content requirement forces the domestic firm to supply a minimum quantity into its own
market. It looks like the most direct way to build domestic capability.

In [ ]:
rows = []
clear_policy(); rows.append(run_policy('no policy'))
for lv in [40.0, 70.0]:
    clear_policy(); set_local_min(lv, market='R2')
    rows.append(run_policy(f'local min {lv:.0f} in R2'))
clear_policy()
pd.DataFrame(rows)

**At 70 units the policy hurts the firm it was designed to protect.** R2's profit *falls* from 6,713
to 5,289, while R1's *rises* to 11,873.

The mechanism is worth understanding, because it is not obvious. R2's profit-maximising allocation
spreads its output across both markets — its home market and, despite the transport premium, R1's
larger one. Forcing a floor on home-market sales pushes it away from that allocation. It sells more
at home, depressing its own home price, and withdraws from R1's market where it had been earning a
better margin. R1 then faces less competition abroad and gains.

**A quantity mandate is not the same instrument as protection.** A tariff or quota raises a rival's
cost; a local content floor constrains *your own* firm's optimisation. If the constraint binds in a
direction the firm did not want, it destroys value on the domestic side of the ledger. Note also that
at 40 units the floor barely binds and almost nothing changes — these policies have sharp thresholds
rather than smooth effects.

## 6. Can a tariff restore investment against a committed leader?

This is the question Part 4d left open. Under Stackelberg the leader commits a large quantity, which
suppresses the follower's **capacity expansion** — the durable form of entry deterrence. Can a tariff
undo it?

In [ ]:
FOLLOWER, LEADER = 'R2', 'R1'

# --- follower's marginal cost per unit DELIVERED to each market -------------
def follower_marginal_cost():
    e_m, e_p, e_f = ETA['MINE', -1, 0], ETA['PROC', -1, 0], ETA['MFG', -1, 0]
    thr_f = 1.0/e_f
    thr_p = thr_f/e_p
    thr_m = thr_p/e_m
    chain = (OPEX['MFG', FOLLOWER]*thr_f + OPEX['PROC', FOLLOWER]*thr_p
             + OPEX['MINE', FOLLOWER]*thr_m)
    return {rt: chain + TRANSPORT[FOLLOWER, rt] for rt in REGIONS}

# follower's inherited deliverable capacity, and the annualised cost of adding more
def follower_legacy(p):
    return LEGACY_CAP['MFG', FOLLOWER]*ETA['MFG', -1, p] if START[p] <= LEGACY_RET['MFG', FOLLOWER] else 0.0

CAP_COST = sum(MU['MFG', v] for v in P)/len(P)*(UNIT['MFG', FOLLOWER]*1.0) + 4.0
BIG_Q, BIG_L = 1200.0, 400.0

In [ ]:
NQ = 6          # grid points for the leader's quantity (binary-selected)


def stackelberg(learning='both', mipgap=0.01, env=None, deter=True, nq=NQ):
    """Single-level MPEC.

    The leader's revenue contains -B*qF*qL, a product of two decision variables.
    We keep it EXACT by restricting the leader's quantity to a finite grid chosen by
    binaries: qL = sum_k S_k * bq_k with sum_k bq_k = 1. Then qF*qL = sum_k S_k*(qF*bq_k),
    and each qF*bq_k is a continuous-times-binary product, which linearises exactly.
    deter=False drops the follower entirely (leader as monopolist)."""
    m = gp.Model(env=env) if env is not None else gp.Model()
    m.Params.OutputFlag = 0
    m.Params.MIPGap = mipgap

    L = add_region(m, LEADER, learning)
    qL = L['sale']
    c_f = follower_marginal_cost()
    KQ = list(range(nq))

    # leader quantity on a binary-selected grid
    bq = m.addVars(REGIONS, P, KQ, vtype=GRB.BINARY, name='bq')
    GRID = {}
    for rt in REGIONS:
        for p in P:
            smax = A_INT[rt, p] / B_SLP[rt, p]
            GRID[rt, p] = [smax * k / (nq - 1) for k in KQ]
            m.addConstr(bq.sum(rt, p, '*') == 1, name=f'gsel_{rt}_{p}')
            m.addConstr(qL[rt, p] == gp.quicksum(GRID[rt, p][k] * bq[rt, p, k] for k in KQ),
                        name=f'gq_{rt}_{p}')

    if deter:
        qF = m.addVars(REGIONS, P, lb=0.0, ub=BIG_Q, name='qF')
        Cap = m.addVar(lb=0.0, ub=BIG_Q, name='CapF')
        lam = m.addVars(P, lb=0.0, name='lam')
        nu = m.addVars(REGIONS, P, lb=0.0, name='nu')
        mcap = m.addVar(lb=0.0, name='mcap')
        yc = m.addVars(P, vtype=GRB.BINARY, name='yc')
        zq = m.addVars(REGIONS, P, vtype=GRB.BINARY, name='zq')
        ycap = m.addVar(vtype=GRB.BINARY, name='ycap')
        slack = m.addVars(P, lb=0.0, name='slk')

        # follower primal feasibility
        m.addConstrs((qF.sum('*', p) + slack[p] == follower_legacy(p) + Cap for p in P),
                     name='fcap')
        # follower stationarity
        m.addConstrs((OMEGA[p]*(A_INT[rt, p] - B_SLP[rt, p]*(2*qF[rt, p] + qL[rt, p])
                                - c_f[rt]) - lam[p] + nu[rt, p] == 0
                      for rt in REGIONS for p in P), name='stat_q')
        m.addConstr(-CAP_COST + gp.quicksum(lam[p] for p in P) + mcap == 0, name='stat_cap')
        # complementarity (big-M)
        m.addConstrs((lam[p] <= BIG_L*yc[p] for p in P), name='cc1')
        m.addConstrs((slack[p] <= BIG_Q*(1-yc[p]) for p in P), name='cc2')
        m.addConstrs((nu[rt, p] <= BIG_L*zq[rt, p] for rt in REGIONS for p in P), name='cc3')
        m.addConstrs((qF[rt, p] <= BIG_Q*(1-zq[rt, p]) for rt in REGIONS for p in P), name='cc4')
        m.addConstr(mcap <= BIG_L*ycap, name='cc5')
        m.addConstr(Cap <= BIG_Q*(1-ycap), name='cc6')

        # exact linearisation of w[rt,p,k] = qF[rt,p] * bq[rt,p,k]
        w = m.addVars(REGIONS, P, KQ, lb=0.0, name='w')
        m.addConstrs((w[rt, p, k] <= BIG_Q*bq[rt, p, k]
                      for rt in REGIONS for p in P for k in KQ), name='w1')
        m.addConstrs((w[rt, p, k] <= qF[rt, p]
                      for rt in REGIONS for p in P for k in KQ), name='w2')
        m.addConstrs((w[rt, p, k] >= qF[rt, p] - BIG_Q*(1-bq[rt, p, k])
                      for rt in REGIONS for p in P for k in KQ), name='w3')
    else:
        qF, Cap = None, None

    # leader revenue = A*qL - B*qL^2 - B*qF*qL, all linear on the grid
    rev = gp.LinExpr()
    for rt in REGIONS:
        for p in P:
            for k in KQ:
                Sk = GRID[rt, p][k]
                rev += OMEGA[p]*(A_INT[rt, p]*Sk - B_SLP[rt, p]*Sk*Sk)*bq[rt, p, k]
                if deter:
                    rev -= OMEGA[p]*B_SLP[rt, p]*Sk*w[rt, p, k]

    m.setObjective(rev - L['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._L, m._qL, m._qF, m._Cap, m._rev = L, qL, qF, Cap, rev
    return m

In [ ]:
rows = []
for t in [0.0, 3.0, 6.0, 10.0]:
    clear_policy(); set_tariff(t, on_imports_to='R2')
    mm = stackelberg(learning='both', env=ENV)
    if mm.SolCount == 0:
        rows.append(dict(tariff=t, status='no solution')); continue
    rows.append(dict(tariff=t, leader_profit=round(mm.ObjVal, 1),
                     leader_qty=round(sum(mm._qL[rt, p].X for rt in REGIONS for p in P), 1),
                     follower_qty=round(sum(mm._qF[rt, p].X for rt in REGIONS for p in P), 1),
                     follower_capacity=round(mm._Cap.X, 2)))
clear_policy()
pd.DataFrame(rows)

**Yes — but only above a threshold, and the threshold is high.**

Read the `follower_capacity` column. It sits at **60.45 for tariffs of 0, 3 and 6** — completely
unmoved. The leader simply absorbs the duty and holds its committed quantity at 1,469, and the
follower's investment calculus does not change at all. Only at a tariff of **10** does the leader's
commitment break down (quantity falls 1,469 → 1,002) and the follower's capacity jump to 88.27.

Two things follow, and both matter for policy design.

**Deterrence is robust to moderate intervention.** A committed incumbent can absorb a substantial
tariff without changing the quantity that does the deterring. Policies calibrated to shift *trade
flows* may leave *investment incentives* untouched — and investment is what the entrant actually
needs.

**The response is a step, not a slope.** Nothing, nothing, nothing, then a jump. Any policy evaluated
by interpolating between a few tariff levels would have concluded the instrument was useless. This is
the lumpy-investment structure of the whole series reappearing at the policy level: with indivisible
capacity, responses come in discrete jumps and thresholds are easy to miss.

The tariff that finally works also costs the leader 36% of its profit, so "effective" and
"proportionate" are not the same thing.

## 7. Summary

| Instrument | Protects the domestic firm? | Government revenue | Total welfare |
|---|---|---|---|
| Tariff | yes, strongly | yes, but plateaus | falls |
| Quota | yes, comparably | **none** | falls **more** |
| Local content | **can backfire** | none | roughly flat, then falls |

### Findings

- **Tariffs redistribute and shrink.** Deadweight loss exceeds revenue collected at every level tested.
- **Quotas are dominated by tariffs.** Same distortion, but the scarcity rent goes uncollected.
- **Local content can hurt its intended beneficiary** by forcing the domestic firm away from its
  preferred allocation. It constrains your own firm rather than raising a rival's cost.
- **Entry deterrence is robust to moderate tariffs.** Follower capacity is flat to a tariff of 6 and
  only jumps at 10.
- **Policy responses are lumpy.** Thresholds, not slopes — a consequence of indivisible investment.

### Limitations, stated plainly

- **Welfare here is narrow.** No supply security, employment, emissions or strategic autonomy — the
  things minerals policy usually exists to address. A welfare-reducing tariff may still be sound.
- **Policy is exogenous.** No retaliation, no strategic tariff-setting. Real trade policy is itself a
  game; modelling it would require a third level.
- **One instrument at a time.** Combinations may interact non-additively, especially given the
  threshold behaviour in §6.
- **The follower cannot make lumpy investments** in the Stackelberg section — inherited from 4d,
  where continuity is what makes the KKT reformulation valid.

### Things to try

- Tariffs on *both* directions, and asymmetric pairs — a crude retaliation experiment
- `set_tariff` combined with `set_local_min` — do they reinforce or cancel?
- A tariff applied only in early periods (infant-industry protection) and then removed: does the
  learning channel make the gain durable after removal?
- `learning='capacity'` throughout — how much of the deterrence effect was the production-learning
  feedback?